In [ ]:
# Diagnosztika

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time

url = "https://www.tippmixpro.hu/hu/elo/i/elo-esemenyek/100/league-of-legends-lol/vilag/lol-world-championship-2025/gen-g-esports-kt-rolster/285417910678360064/all"

chrome_options = Options()
# chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=chrome_options)

try:
    print("Oldal betöltése...")
    driver.get(url)

    # Cookie elfogadás
    try:
        cookie_btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
        )
        cookie_btn.click()
        time.sleep(1)
    except Exception:
        pass

    # Váltás az első iframe-re
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "iframe")))
    iframe = driver.find_elements(By.TAG_NAME, "iframe")[0]
    driver.switch_to.frame(iframe)
    time.sleep(2)

    # Article elemek begyűjtése
    articles = driver.find_elements(By.TAG_NAME, "article")
    print(f"{len(articles)} market található.")

    # Az első 2 article teljes HTML-je
    for i, art in enumerate(articles[:2]):
        html = art.get_attribute("outerHTML")
        print(f"\n--- ARTICLE {i+1} HTML snippet ---")
        print(html[:800].replace("\n", "") + " ...")

        with open(f"article_{i+1}.html", "w", encoding="utf-8") as f:
            f.write(html)

    print("\nKimentve: article_1.html, article_2.html")

except Exception as e:
    print("Hiba:", e)
finally:
    driver.quit()


In [ ]:
# Tippmix live iframe scrape

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time

url = "https://www.tippmixpro.hu/hu/fogadas/i/esemenyek/100/league-of-legends-lol/vilag/emea-masters-summer/karmine-corp-blue-los-heretics/284726865528393728/palyak"

chrome_options = Options()
#chrome_options.add_argument("--headless=new")  # ha nem akarsz böngészőt látni, vedd ki a kommentet
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=chrome_options)

try:
    print("Oldal betöltése...")
    driver.get(url)

    # Cookie elfogadás
    try:
        cookie_btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
        )
        cookie_btn.click()
        time.sleep(1)
    except Exception:
        pass

    # Váltás az első iframe-re
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "iframe")))
    iframe = driver.find_elements(By.TAG_NAME, "iframe")[0]
    driver.switch_to.frame(iframe)
    time.sleep(2)

    # Article elemek begyűjtése
    articles = driver.find_elements(By.TAG_NAME, "article")
    print(f"{len(articles)} market található.")

    for art in articles:
        try:
            # Market neve
            legend_el = art.find_element(By.CLASS_NAME, "Market__CollapseText")
            legend = legend_el.get_attribute("title") or legend_el.text.strip()
            print(f"\n=== {legend} ===")

            # Odds gombok keresése
            odds_buttons = art.find_elements(By.CLASS_NAME, "OddsButton")
            for btn in odds_buttons:
                try:
                    name_el = btn.find_element(By.CLASS_NAME, "OddsButton__Text")
                    odds_el = btn.find_element(By.CLASS_NAME, "OddsButton__Odds")
                    name = name_el.text.strip()
                    odds = odds_el.text.strip()
                    print(f"{name}: {odds}")
                except Exception:
                    # vannak olyan gombok, ahol nincs név (pl. over/under)
                    odds_el = btn.find_element(By.CLASS_NAME, "OddsButton__Odds")
                    odds = odds_el.text.strip()
                    print(f"Odds: {odds}")
        except Exception:
            continue

except Exception as e:
    print("Hiba:", e)
finally:
    driver.quit()
